# 6.1 章节介绍：全量化 FA 开发

第 5 章我们以七阶段渐进路线从 0 到 1 构建了**非量化** FA Kernel：Q、K、V 全程 FP16/BF16，Softmax 在 FP32 上累加。本章把这块骨架保留，换掉其中的「数据格式」——进入**全量化** FA 的世界。

> 「全量化」指左、右矩阵（激活与权重 / 这里主要是 Q 与 KV Cache）均做量化的模式，与之相对的「伪量化」是低精度存储、高精度计算。

## 为什么 FA 需要量化

回忆第 2 章的搬运账单：FA 的复杂度从 O(N²d) 降到 O(N²d²/block)，但**每个 K/V 元素要被搬运的次数没有变少**。推理 Decode 场景更是典型的带宽受限（memory-bound）：

| 项 | BF16 KV Cache | MXFP8 KV Cache | 收益 |
|--|--|--|--|
| 显存占用 | 1× | ~1/2（数据）+ 1/32（scale） | ≈ 减半 → 能放更长的序列 / 更大 batch |
| MTE2 搬运量 | 1× | ~1/2 | 带宽受限场景下直接提速 |
| Cube 吞吐（若硬件支持 FP8 Mmad） | 1× | ~2× | 计算受限场景下额外收益 |

> KV Cache 是推理显存的大头：`2 × B × S × N × D × bytes`，序列一长首先爆的就是它。量化 KV Cache 是「省带宽 + 省显存」双收的一刀。

## 本章学习路线

| 小节 | 主题 | 回答的问题 |
|--|--|--|
| [6.2 数据格式](06.02_data_formats.ipynb) | FP8 家族 / E8M0 / HiF8 | 8-bit 里到底塞了什么？为什么 FP8×FP8 要用 FP32 累加？ |
| [6.3 量化原理](06.03_quant_principle.ipynb) | scale / 零点 / 对称非对称 / 静态动态 | 量化数学上是「舍入 + 缩放」，误差从哪来？ |
| [6.4 量化方式](06.04_quant_modes.ipynb) | T/C/K/G/B 粒度 / 全量化组合 / mx 量化 / 随路量化 | scale 按什么粒度切？MXFP8 是什么？ |
| [6.5 FA 支持 MXFP8 量化方案](06.05_fa_mxfp8.ipynb) | 块浮点：E4M3 数据 + E8M0 scale | 第 5 章的 Kernel 要改哪几处？ |
| [6.6 FA 支持 HiP8 量化方案](06.06_fa_hif8.ipynb) | 单数据格式：锥形精度 | 与 MXFP8 的 diff 与取舍 |
| [6.7 章节测验](06.07_chapter_test.ipynb) | 综合测验 | 检验本章掌握程度 |

## 与第 5 章的关系

全量化 FA 的 Kernel 骨架（四拍循环、分核、乒乓、preload、mask、尾块、负载均衡）**全部继承第 5 章**。量化改变的只有三处：

1. **搬入**：GM 里的数据是 8-bit + scale，搬进来的东西变了；
2. **计算前**：多一步「随路反量化」（或硬件直吃 FP8）；
3. **计算后**：scale 要乘回正确的位置（这一点最容易写错，见 6.5 节）。

所以本章的重心不是「再写一个 Kernel」，而是**数据通路的变化**——把第 5 章的 Kernel 当作底座，理解量化给数据通路打的补丁。

## 前置知识

- [第5章 非量化 FA 开发](../05_flash_attn_non_quantized/README.md)：四拍循环与 Kernel 骨架
- [第1章 昇腾开发基础](../01_ascend_dev_basics/README.md)：MTE2/MTE3 搬运指令、Cube/Vector 分工
- [第3章 FA 算子代码结构](../03_fa_code_architecture/README.md)：Tiling / Kernel 两层结构